In [1]:
import pickle
import numpy as np
import tensorflow as tf

print("Loading Multi-Agent System...")

# Load Agent 1 (RNN)
agent1_rnn = tf.keras.models.load_model("../models/agent1_rnn.h5", compile=False)
print("✅ Agent 1 (RNN) loaded.")

# Load Agent 2 (Random Forest)
with open("../models/agent2_rf.pkl", "rb") as f:
    agent2_rf = pickle.load(f)
print("✅ Agent 2 (Random Forest) loaded.")

# Load Agent 3 (Logistic Regression)
with open("../models/agent3_lr.pkl", "rb") as f:
    agent3_lr = pickle.load(f)
print("✅ Agent 3 (Logistic Regression) loaded.")

print("\n🎉 All agents are ready for the Coordinator!")

Loading Multi-Agent System...
✅ Agent 1 (RNN) loaded.
✅ Agent 2 (Random Forest) loaded.
✅ Agent 3 (Logistic Regression) loaded.

🎉 All agents are ready for the Coordinator!


In [2]:
def credit_scoring_pipeline(customer_cc_data, customer_hc_data):
    """
    Coordinates the 3 agents to make a final credit decision.
    """
    # 1. Agent 1 (RNN) - Transaction Analysis
    # Reshape the data to (1, 1, 30) for the RNN
    cc_input = customer_cc_data.reshape((1, 1, customer_cc_data.shape[0]))
    prob_agent1 = agent1_rnn.predict(cc_input, verbose=0)[0][0]
    
    # 2. Agent 2 (Random Forest) - Profile Analysis
    # Ensure the data is 2D (1, 79) for the Random Forest
    hc_input = customer_hc_data.reshape(1, -1) 
    prob_agent2 = agent2_rf.predict_proba(hc_input)[0][1]
    
    # 3. Agent 3 (Logistic Regression) - Final Decision
    # Combine Agent 1 and Agent 2 predictions
    meta_features = np.array([[prob_agent1, prob_agent2]])
    prob_final = agent3_lr.predict_proba(meta_features)[0][1]
    
    # Make final decision (threshold at 50%)
    decision = "APPROVED ✅" if prob_final < 0.5 else "REJECTED ❌"
    
    return {
        "Agent 1 (RNN) Fraud Risk": f"{prob_agent1*100:.2f}%",
        "Agent 2 (RF) Default Risk": f"{prob_agent2*100:.2f}%",
        "Final Default Risk (Agent 3)": f"{prob_final*100:.2f}%",
        "Final Decision": decision
    }

print("✅ Coordinator function created successfully!")

✅ Coordinator function created successfully!


In [3]:
import pandas as pd

# Load the cleaned test data to simulate a customer
with open("../data/clean/cc_data.pkl", "rb") as f:
    _, X_cc_test, _, _ = pickle.load(f)
with open("../data/clean/hc_data.pkl", "rb") as f:
    _, X_hc_test, _, _, _ = pickle.load(f)

# Pick a random customer from the test set (e.g., index 42)
customer_idx = 42
customer_transactions = X_cc_test.values[customer_idx]
customer_profile = X_hc_test[customer_idx]

print(f"--- Simulating Credit Application for Customer #{customer_idx} ---\n")

# Run the pipeline
result = credit_scoring_pipeline(customer_transactions, customer_profile)

# Print the results nicely
for key, value in result.items():
    print(f"{key}: {value}")

--- Simulating Credit Application for Customer #42 ---

Agent 1 (RNN) Fraud Risk: 0.00%
Agent 2 (RF) Default Risk: 44.01%
Final Default Risk (Agent 3): 44.10%
Final Decision: APPROVED ✅
